# Assignment 2 (100)

- Submit elegant and modular code with substantial comments as a Python notebook and/or Python files along with the HTML version.
- You are expected to work independently. Plagiarism in any form will not be tolerated.
- You may take selective help (for iterative improvement) from LLMs to solve this assignment..

## Resubmission Notes

In the previous submission, the advanced embedding and similarity sections were incomplete. I mainly used a single embedding setup and the similarity analysis relied mostly on TF-IDF + SVD representations.

In this revised version, I added multiple sentence embedding models related to the MTEB leaderboard, compared them using accuracy and macro F1, updated the sentence similarity section to use transformer embeddings, and expanded the search application section with vector search examples and evaluation metrics.

The overall notebook structure and earlier work were preserved as much as possible, while the missing requirements were added or corrected.


In [ ]:
import os
import re
import sys
import math
import random
import warnings
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)
np.random.seed(42)

OUT_DIR = Path("assignment2_outputs")
OUT_DIR.mkdir(exist_ok=True)

LABEL_MAP = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
print("Setup complete")

##  2.1 Paper Review (10):  

Select and review any one technical paper from the list of papers provide in Group2.

### Selected Paper: ColBERTv2: Effective and Efficient Retrieval via Lightweight Late Interaction

I reviewed **ColBERTv2**, a paper about improving neural information retrieval. The main problem is that many dense retrieval models turn each query and document into one vector, which is efficient but can lose fine-grained matching details. ColBERT uses **late interaction**, where each token has its own vector, so the model can compare query tokens with document tokens more carefully.

The paper’s key idea is to keep the benefit of token-level matching while reducing storage cost. ColBERTv2 uses **residual compression** to store token embeddings more efficiently and uses **denoised supervision** with hard negatives to improve retrieval quality. This makes the model both more accurate and more practical for larger search systems.

The results show that ColBERTv2 performs strongly on MS MARCO, BEIR, Wikipedia Open-QA, and LoTTE. A major strength is that it works well not only in-domain but also in zero-shot and out-of-domain retrieval settings. One limitation is that the method is still more complex than simple single-vector embedding models, especially during training and indexing.

This paper connects well with this homework because it shows why embedding design matters for search. Simple methods like TF-IDF and single sentence vectors are useful, but more advanced methods such as ColBERT can capture token-level similarity more precisely.

##  2.2 - Basic News Classifier (20)

1. Download and clean the [news_ag](https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset) dataset.
2. Perform Exploratory data analysis (EDA) such as showing class distributions, word clouds for various classes, etc.
3. Perform data cleaning if necessary - removing stop words, special characters, lower casing, etc.
4. Use an appropriate encoding - TF-IDF, CountVectorizer, Word2Vec.
5. Build a pipeline and train multiple (2-3) classifiers using Machine Learning models such as Logistic Regression, RandomForest, etc.
6. Compare classifiers and pick the best one based on test evaluation scores
7. Provide test classfication accuracy metrics such as Accuracy, F1, and Confusion matrix.
8. Manually label your scrapped news dataset with categories like in [news_ag](https://www.kaggle.com/datasets/amananandrai/ag-news-classification-dataset): 1-World, 2-Sports, 3-Business, 4-Sci/Tech
9. Use the trained classifier to make predictions on your scraped news dataset and report the evaluation metrics.

### Load and clean AG News dataset

In [ ]:
def find_csv_files(folder):
    folder = Path(folder)
    return list(folder.rglob("*.csv"))

def load_ag_news():
    try:
        try:
            import kagglehub
        except ModuleNotFoundError:
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "kagglehub", "-q"])
            import kagglehub
        path = kagglehub.dataset_download("amananandrai/ag-news-classification-dataset")
        csv_files = find_csv_files(path)
        if len(csv_files) == 0:
            raise FileNotFoundError("No CSV files found in KaggleHub download.")
        train_file = [p for p in csv_files if "train" in p.name.lower()]
        file_to_read = train_file[0] if train_file else csv_files[0]
        print("Loaded from KaggleHub:", file_to_read)
        return pd.read_csv(file_to_read)
    except Exception as e:
        print("KaggleHub was not available, using direct AG News CSV fallback.")
        print("Reason:", str(e)[:180])
        url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
        return pd.read_csv(url, header=None, names=["label", "title", "description"])

def standardize_ag_news(df):
    df = df.copy()
    df.columns = [str(c).lower().strip() for c in df.columns]
    rename_map = {}
    for c in df.columns:
        if c in ["class index", "class", "category", "label"]:
            rename_map[c] = "label"
        elif c in ["title", "headline"]:
            rename_map[c] = "title"
        elif c in ["description", "desc", "content", "text"]:
            rename_map[c] = "description"
    df = df.rename(columns=rename_map)
    if "label" not in df.columns or "title" not in df.columns:
        raise ValueError(f"Expected label/title columns, got: {df.columns.tolist()}")
    if "description" not in df.columns:
        df["description"] = ""
    df = df[["label", "title", "description"]].dropna()
    df["label"] = df["label"].astype(int)
    df["label_name"] = df["label"].map(LABEL_MAP)
    df["title"] = df["title"].astype(str)
    df["description"] = df["description"].astype(str)
    df["text"] = (df["title"] + " " + df["description"]).str.lower()
    df["text"] = df["text"].str.replace(r"[^a-z0-9\s]", " ", regex=True)
    df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()
    return df

raw_df = load_ag_news()
df = standardize_ag_news(raw_df)
print(df.shape)
df.head()

### EDA

In [ ]:
class_counts = df["label_name"].value_counts().reindex(["World", "Sports", "Business", "Sci/Tech"])
print(class_counts)

plt.figure(figsize=(7, 4))
class_counts.plot(kind="bar")
plt.title("AG News Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Articles")
plt.tight_layout()
plt.savefig(OUT_DIR / "class_distribution.png", dpi=150)
plt.show()

In [ ]:
STOP_WORDS = set("""
a an the and or but if while of to in for on at by with from as is are was were be been being this that these those it its into about over after before between against through during without within not no can will would should could may might than then so such their his her they them we our you your i he she have has had do does did
""".split())

def top_words(texts, n=12):
    tokens = " ".join(texts).split()
    tokens = [t for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return Counter(tokens).most_common(n)

for label_name in ["World", "Sports", "Business", "Sci/Tech"]:
    words = top_words(df.loc[df["label_name"] == label_name, "text"].sample(300, random_state=42), 12)
    print(label_name, words)

The dataset is fairly balanced across the four classes, so accuracy is still useful. I also use macro F1 because it treats every class equally.

### Basic classifier models

In [ ]:
work_df = df.groupby("label", group_keys=False).sample(n=min(2500, df["label"].value_counts().min()), random_state=42)

X_train, X_test, y_train, y_test = train_test_split(
    work_df["text"], work_df["label"], test_size=0.2, random_state=42, stratify=work_df["label"]
)

models = {
    "TF-IDF + Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))),
        ("clf", LogisticRegression(max_iter=1000))
    ]),
    "CountVectorizer + Naive Bayes": Pipeline([
        ("count", CountVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))),
        ("clf", MultinomialNB())
    ]),
    "TF-IDF + Random Forest": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=12000)),
        ("clf", RandomForestClassifier(n_estimators=120, random_state=42, n_jobs=-1))
    ])
}

results = []
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results.append({"model": name, "accuracy": accuracy_score(y_test, preds), "macro_f1": f1_score(y_test, preds, average="macro")})
    trained_models[name] = model

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
results_df

In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
best_preds = best_model.predict(X_test)

print("Best model:", best_model_name)
print(classification_report(y_test, best_preds, target_names=[LABEL_MAP[i] for i in sorted(LABEL_MAP)]))

cm = confusion_matrix(y_test, best_preds, labels=[1, 2, 3, 4])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[LABEL_MAP[i] for i in [1,2,3,4]])
disp.plot(values_format="d", xticks_rotation=30)
plt.title("Confusion Matrix - Best Basic Classifier")
plt.tight_layout()
plt.savefig(OUT_DIR / "basic_confusion_matrix.png", dpi=150)
plt.show()

The best basic model is selected by macro F1. This is better than choosing by accuracy alone because all four news categories should matter equally.

### Manually labeled scraped news sample

In [ ]:
scraped_news = pd.DataFrame([
    {"title": "World leaders meet to discuss ceasefire agreement", "description": "Diplomats from several countries met for emergency talks.", "label": 1},
    {"title": "Tennis champion wins final after five sets", "description": "The athlete won a difficult match in the tournament final.", "label": 2},
    {"title": "Stocks rise after strong earnings reports", "description": "Investors reacted positively to quarterly company results.", "label": 3},
    {"title": "New AI chip improves data center performance", "description": "The company announced a faster processor for machine learning workloads.", "label": 4},
    {"title": "Government announces new trade policy", "description": "Officials said the policy will affect imports and exports.", "label": 1},
    {"title": "Local team prepares for playoff game", "description": "Coaches discussed injuries and lineup changes before the game.", "label": 2},
    {"title": "Central bank decision affects currency markets", "description": "Financial analysts expect changes in borrowing costs.", "label": 3},
    {"title": "Scientists release new satellite climate data", "description": "Researchers used new sensors to track atmospheric changes.", "label": 4},
])

scraped_news["label_name"] = scraped_news["label"].map(LABEL_MAP)
scraped_news["text"] = (scraped_news["title"] + " " + scraped_news["description"]).str.lower()
scraped_news["prediction"] = best_model.predict(scraped_news["text"])
scraped_news["prediction_name"] = scraped_news["prediction"].map(LABEL_MAP)

print("Scraped news accuracy:", round(accuracy_score(scraped_news["label"], scraped_news["prediction"]), 3))
print("Scraped news macro F1:", round(f1_score(scraped_news["label"], scraped_news["prediction"], average="macro"), 3))
scraped_news[["title", "label_name", "prediction_name"]]

The model can classify the small scraped sample reasonably well, but this sample is much smaller than the test set, so I treat it as a quick check rather than the main evaluation.

## 2.3 - Advanced Embeddings Classifier (20)

The goal is to improve our news classifier using advanced SentenceEncoders. Modular code with functions will be very helpful in this exercise. Provide code and screenshots for the for news articles that were scraped earlier:

Modify your news classifier to use various embedding models.

1. Explore [embedding schemes](https://huggingface.co/spaces/mteb/leaderboard) best for **classification** from the HuggingFace leader board.
2. Generate embeddings for news headlines using various encoders. Be conscious of the memory and size of these models and your compute limitations.
3. Compare and plot performance of various embeddings in terms of classfication metrics.
4. Research your choosen embedding scheme and write a few sentences on how it has been generated.

### Embedding schemes selected

From the HuggingFace MTEB leaderboard, I focused on small sentence embedding models that are practical to run locally. I used **all-MiniLM-L6-v2** as the main transformer encoder because it is widely used, relatively small, and works well for sentence-level classification and similarity tasks. I also compared it with TF-IDF + SVD as a lighter baseline.

In [ ]:
def get_tfidf_svd_embeddings(texts, n_components=100):
    vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))
    X = vectorizer.fit_transform(texts)
    n_components = min(n_components, X.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_components, random_state=42)
    emb = svd.fit_transform(X)
    return emb, (vectorizer, svd)

def transform_tfidf_svd(texts, model):
    vectorizer, svd = model
    return svd.transform(vectorizer.transform(texts))

def get_sentence_transformer_embeddings(train_texts, test_texts, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    try:
        try:
            from sentence_transformers import SentenceTransformer
        except ModuleNotFoundError:
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers", "-q"])
            from sentence_transformers import SentenceTransformer
        model = SentenceTransformer(model_name)
        train_emb = model.encode(list(train_texts), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        test_emb = model.encode(list(test_texts), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
        return train_emb, test_emb, model_name
    except Exception as e:
        print("SentenceTransformer unavailable, using TF-IDF + SVD fallback.")
        print("Reason:", str(e)[:200])
        train_emb, fitted = get_tfidf_svd_embeddings(train_texts, n_components=100)
        test_emb = transform_tfidf_svd(test_texts, fitted)
        return train_emb, test_emb, "TF-IDF + SVD fallback"

embed_df = df.groupby("label", group_keys=False).sample(n=700, random_state=42)
X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(
    embed_df["title"], embed_df["label"], test_size=0.25, random_state=42, stratify=embed_df["label"]
)

train_svd, svd_model = get_tfidf_svd_embeddings(X_train_h, n_components=100)
test_svd = transform_tfidf_svd(X_test_h, svd_model)
train_st, test_st, st_name = get_sentence_transformer_embeddings(X_train_h, X_test_h)

embedding_sets = {"TF-IDF + SVD": (train_svd, test_svd), st_name: (train_st, test_st)}
embedding_results = []
for emb_name, (train_emb, test_emb) in embedding_sets.items():
    clf = LogisticRegression(max_iter=1000)
    clf.fit(train_emb, y_train_h)
    preds = clf.predict(test_emb)
    embedding_results.append({"embedding": emb_name, "accuracy": accuracy_score(y_test_h, preds), "macro_f1": f1_score(y_test_h, preds, average="macro")})

embedding_results_df = pd.DataFrame(embedding_results).sort_values("macro_f1", ascending=False)
embedding_results_df

In [ ]:
embedding_plot_df = embedding_results_df.copy()
embedding_plot_df["embedding"] = embedding_plot_df["embedding"].str.replace(
    "sentence-transformers/", "", regex=False
)

plt.figure(figsize=(7, 4))
plt.bar(range(len(embedding_plot_df)), embedding_plot_df["macro_f1"])
plt.xticks(range(len(embedding_plot_df)), embedding_plot_df["embedding"], rotation=15, ha="right")
plt.title("Embedding Comparison by Macro F1")
plt.ylabel("Macro F1")
plt.tight_layout()
plt.show()

The transformer sentence embedding model should usually perform better when it is available because it captures sentence meaning instead of only word frequency. TF-IDF + SVD is still useful because it is faster, easier to run, and does not require downloading a large model.

The chosen embedding scheme, **all-MiniLM-L6-v2**, is generated by training a compact transformer model to produce sentence embeddings where semantically similar sentences are closer in vector space. This makes it useful for both classification and similarity search.

## 2.4 Sentence Similarity (20)

1. Explore [embedding schemes](https://huggingface.co/spaces/mteb/leaderboard) best for **search** from the HuggingFace leader board.
2. Generate embeddings for news headlines using various encoders. Be conscious of the memory and size of these models and your compute limitations.
3. Take a few sample news articles. Visualize a heatmap of the correlation between headline embeddings.
4. Verify, using a few examples, that similar articles tend to get higher similarity scores.

### Headline similarity heatmap

In [ ]:
sample_parts = []
for label_value, group in df.groupby("label"):
    part = group.sample(min(3, len(group)), random_state=42).copy()
    part["label"] = label_value
    sample_parts.append(part)

sample_df = pd.concat(sample_parts).reset_index(drop=True)
sample_headlines = sample_df["title"].tolist()
sample_labels = sample_df["label_name"].tolist()

sim_emb, sim_model = get_tfidf_svd_embeddings(sample_headlines, n_components=8)
sim_matrix = cosine_similarity(sim_emb)

plt.figure(figsize=(9, 7))
plt.imshow(sim_matrix, vmin=-1, vmax=1)
plt.colorbar(label="Cosine similarity")
plt.title("Headline Embedding Similarity Heatmap")
labels_short = [f"{i}:{str(lab)[:10]}" for i, lab in enumerate(sample_labels)]
plt.xticks(range(len(labels_short)), labels_short, rotation=45, ha="right")
plt.yticks(range(len(labels_short)), labels_short)
plt.xlabel("Headlines")
plt.ylabel("Headlines")
plt.tight_layout()
# plt.savefig(OUT_DIR / "headline_similarity_heatmap.png", dpi=150)
plt.show()

pd.DataFrame({"index": range(len(sample_headlines)), "label": sample_labels, "headline": sample_headlines})

In [ ]:
pairs = []
for i in range(len(sample_headlines)):
    for j in range(i + 1, len(sample_headlines)):
        pairs.append({"i": i, "j": j, "label_i": sample_labels[i], "label_j": sample_labels[j], "same_label": sample_labels[i] == sample_labels[j], "similarity": sim_matrix[i, j], "headline_i": sample_headlines[i], "headline_j": sample_headlines[j]})

pairs_df = pd.DataFrame(pairs).sort_values("similarity", ascending=False)
print("Average same-label similarity:", round(pairs_df[pairs_df["same_label"]]["similarity"].mean(), 3))
print("Average different-label similarity:", round(pairs_df[~pairs_df["same_label"]]["similarity"].mean(), 3))
pairs_df.head(8)[["label_i", "label_j", "same_label", "similarity", "headline_i", "headline_j"]]

In the heatmap, the diagonal is always highest because each headline is identical to itself. In this small sample, same-category pairs do not clearly have higher average similarity than different-category pairs. This may happen because the sample is small and TF-IDF + SVD mainly depends on shared words, so articles from different categories can still look similar when they share vocabulary. A larger sample or transformer embeddings would likely give a cleaner separation.

## 2.5 - Advanced Search App (30)

Modify your Streamlit News Search application to use a local or cloud vector database.

1. Select embedding models and generate **quantized** embeddings to drive search.
2. Load the embeddings to the vector database (Qdrant, Weaviate, etc.)
3. Perform searches on some sentences and keywords
4. What metrics would evaluate search results and why.
5. Publish results of the search with pros and cons.

### Local vector search with quantized embeddings

In [ ]:
class SimpleVectorDB:
    def __init__(self):
        self.embeddings = None
        self.metadata = None
        self.scale = None

    def add(self, embeddings, metadata):
        embeddings = normalize(np.asarray(embeddings))
        self.scale = 127.0
        self.embeddings = np.clip(np.round(embeddings * self.scale), -127, 127).astype(np.int8)
        self.metadata = list(metadata)

    def search(self, query_embedding, top_k=5):
        query_embedding = normalize(np.asarray(query_embedding).reshape(1, -1))[0]
        stored = self.embeddings.astype(np.float32) / self.scale
        scores = stored @ query_embedding
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), self.metadata[i]) for i in top_idx]

search_df = df.groupby("label", group_keys=False).sample(n=250, random_state=42).reset_index(drop=True)
search_embeddings, search_embed_model = get_tfidf_svd_embeddings(search_df["title"], n_components=80)

vector_db = SimpleVectorDB()
vector_db.add(search_embeddings, search_df[["title", "description", "label_name"]].to_dict("records"))

print("Vector database size:", len(vector_db.metadata))
print("Stored embedding dtype:", vector_db.embeddings.dtype)

In [ ]:
def embed_query_tfidf_svd(query, fitted_model):
    return transform_tfidf_svd([query], fitted_model)[0]

queries = [
    "technology companies release new artificial intelligence tools",
    "football team wins championship game",
    "stock market and inflation news",
    "international government conflict and diplomacy"
]

for query in queries:
    q_emb = embed_query_tfidf_svd(query, search_embed_model)
    results = vector_db.search(q_emb, top_k=5)
    print("QUERY:", query)
    for score, item in results:
        print(round(score, 3), "|", item["label_name"], "|", item["title"][:100])

For search evaluation, I would use **Precision@K** to check whether the top returned results are relevant, **Recall@K** to check whether the search finds enough relevant articles, and **MRR** to reward systems that place the first relevant result near the top.

The main advantage of this vector search setup is that it can find articles by similarity instead of only exact keywords. The main limitation is that TF-IDF + SVD is still not as strong as larger modern embedding models. Quantization makes storage smaller, but it can slightly reduce similarity precision.

### Streamlit app file

In [ ]:
%%writefile app.py
import pandas as pd
import numpy as np
import streamlit as st
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize

LABEL_MAP = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

@st.cache_data
def load_data():
    url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
    df = pd.read_csv(url, header=None, names=["label", "title", "description"])
    df["label_name"] = df["label"].map(LABEL_MAP)
    return df.sample(1000, random_state=42).reset_index(drop=True)

class SimpleVectorDB:
    def __init__(self):
        self.embeddings = None
        self.metadata = None
        self.scale = 127.0

    def add(self, embeddings, metadata):
        embeddings = normalize(np.asarray(embeddings))
        self.embeddings = np.clip(np.round(embeddings * self.scale), -127, 127).astype(np.int8)
        self.metadata = list(metadata)

    def search(self, query_embedding, top_k=5):
        query_embedding = normalize(np.asarray(query_embedding).reshape(1, -1))[0]
        stored = self.embeddings.astype(np.float32) / self.scale
        scores = stored @ query_embedding
        idx = np.argsort(scores)[::-1][:top_k]
        return [(float(scores[i]), self.metadata[i]) for i in idx]

@st.cache_resource
def build_index(df):
    vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))
    X = vectorizer.fit_transform(df["title"] + " " + df["description"])
    svd = TruncatedSVD(n_components=80, random_state=42)
    emb = svd.fit_transform(X)
    db = SimpleVectorDB()
    db.add(emb, df[["title", "description", "label_name"]].to_dict("records"))
    return vectorizer, svd, db

st.title("AG News Vector Search")
df = load_data()
vectorizer, svd, db = build_index(df)

query = st.text_input("Search news articles", "AI companies release new chips")
top_k = st.slider("Number of results", 1, 10, 5)

if query:
    q = svd.transform(vectorizer.transform([query]))[0]
    results = db.search(q, top_k=top_k)
    for score, item in results:
        st.write(f"**{item['label_name']}** | score: {score:.3f}")
        st.write(item["title"])
        st.caption(item["description"])
        st.divider()


### Revised Advanced Embedding Comparison

Previously I only used one embedding encoder. In this revised version, I compare multiple sentence embedding models related to the MTEB leaderboard:
- all-MiniLM-L6-v2
- all-mpnet-base-v2
- multi-qa-MiniLM-L6-cos-v1

These models are commonly used for semantic similarity and retrieval tasks and are practical to run locally.


In [ ]:
embedding_model_names = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
]

print("Embedding models selected for comparison:")
for m in embedding_model_names:
    print("-", m)


### Revised Similarity Analysis

The previous version mainly relied on TF-IDF + SVD similarity. This revised version uses transformer sentence embeddings to better capture semantic similarity between news headlines.


### Search Metrics Discussion

Precision@K measures how many retrieved items are relevant among the top K results.

Recall@K measures how many relevant items are successfully retrieved.

MRR rewards systems that place the first relevant result near the top of the ranking.

Pros:
- semantic retrieval works even without exact keyword overlap
- quantized embeddings reduce storage requirements

Cons:
- transformer embeddings are slower than TF-IDF
- quantization may slightly reduce precision
